In [1]:
#import libraries
import psycopg2
import numpy as np
import re
import dash
from dash import dcc, html, Input, Output, dash_table, State
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from dash.exceptions import PreventUpdate
from datetime import datetime
from dash.dependencies import Input, Output, State
import dash_bootstrap_components as dbc


In [2]:
#Reading data from postgresql

# Establish connection
conn = psycopg2.connect(
    host="localhost",
    database="APDV",
    user="postgres",
    password="Nevin"
)

# Define query
sql_query = "SELECT * FROM airquality"

# Execute with parameters
params = ('value',)
airqualitydata = pd.read_sql(sql_query, conn, params=params)

# Display results
print(airqualitydata.head())

# Close connection
conn.close()

   index  state_code  county_code  site_number  parameter_code  poc parameter  \
0      0           1            3           10           44201    1     Ozone   
1      1           1            3           10           44201    1     Ozone   
2      2           1           49         9991           44201    1     Ozone   
3      3           1           51            4           44201    1     Ozone   
4      4           1           51            4           44201    1     Ozone   

    si_id  method_code                                method  ... state_name  \
0       7           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
1       7           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
2   96297           47             INSTRUMENTAL-ULTRA VIOLET  ...    Alabama   
3  104232           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   
4  104232           87  INSTRUMENTAL-ULTRA VIOLET ABSORPTION  ...    Alabama   

   county_name   city_name cbsa_

C:\Users\jismo\AppData\Local\Temp\ipykernel_83948\1500258671.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  airqualitydata = pd.read_sql(sql_query, conn, params=params)


In [3]:
airqualitydata.shape

(3738, 59)

In [4]:
#information
airqualitydata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3738 entries, 0 to 3737
Data columns (total 59 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   index                           3738 non-null   int64  
 1   state_code                      3738 non-null   int64  
 2   county_code                     3738 non-null   int64  
 3   site_number                     3738 non-null   int64  
 4   parameter_code                  3738 non-null   int64  
 5   poc                             3738 non-null   int64  
 6   parameter                       3738 non-null   object 
 7   si_id                           3738 non-null   int64  
 8   method_code                     3738 non-null   int64  
 9   method                          3738 non-null   object 
 10  assessment_date                 3738 non-null   object 
 11  assessment_number               3738 non-null   float64
 12  unit_code                       37

In [5]:
#descriptive statistics
airqualitydata.describe()

,index,state_code,county_code,site_number,parameter_code,poc,si_id,method_code,assessment_number,unit_code,...,lvl10_monitor_concentration,lvl10_assessment_concentration,pqao_code,monitoring_agency_code,performing_agency_code,latitude,longitude,cbsa_code,csa_code,tribal_code
count,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,...,107.000000,107.000000,3738.000000,3738.000000,3738.000000,3738.000000,3738.000000,3424.000000,2627.000000,62.000000
mean,1868.500000,28.809791,75.918406,1014.312199,43371.620385,1.183521,48208.440075,164.764580,1.026217,7.553772,...,193.063675,193.035850,779.007758,756.154093,775.318887,38.524583,-91.406361,30903.457944,343.518462,563.983871
std,1079.211981,15.275349,83.187860,2209.350815,891.179367,0.616017,44839.736261,180.245929,0.226331,0.497167,...,118.816449,120.313015,370.665293,339.934630,363.887953,4.667703,15.870097,10982.756190,126.476128,331.277631
min,0.000000,1.000000,1.000000,1.000000,42101.000000,1.000000,7.000000,9.000000,1.000000,7.000000,...,0.210000,0.212000,13.000000,12.000000,9.000000,19.060655,-159.366240,10220.000000,104.000000,17.000000
25%,934.250000,17.000000,21.000000,8.000000,42401.000000,1.000000,7124.750000,74.000000,1.000000,7.000000,...,161.900000,165.100000,584.000000,584.000000,584.000000,35.503199,-103.273777,19780.000000,216.000000,413.500000
50%,1868.500000,30.000000,59.000000,40.000000,44201.000000,1.000000,15415.000000,87.000000,1.000000,8.000000,...,249.000000,249.000000,768.000000,768.000000,768.000000,39.469219,-87.524047,33500.000000,370.000000,750.000000
75%,2802.750000,40.000000,103.000000,1005.750000,44201.000000,1.000000,96337.500000,100.000000,1.000000,8.000000,...,253.000000,250.000000,1035.000000,1001.000000,1035.000000,41.601899,-78.768924,39900.000000,430.000000,750.000000
max,3737.000000,56.000000,800.000000,9997.000000,44201.000000,9.000000,105404.000000,600.000000,4.000000,8.000000,...,415.000000,416.000000,6532.000000,6532.000000,2368.000000,64.845690,-67.061325,49740.000000,566.000000,905.000000


In [6]:
#percentage of missing value
airqualitydata.isnull().sum()/airqualitydata.shape[0]*100

index                               0.000000
state_code                          0.000000
county_code                         0.000000
site_number                         0.000000
parameter_code                      0.000000
poc                                 0.000000
parameter                           0.000000
si_id                               0.000000
method_code                         0.000000
method                              0.000000
assessment_date                     0.000000
assessment_number                   0.000000
unit_code                           0.000000
unit                                0.000000
lvl1_monitor_concentration         72.953451
lvl1_assessment_concentration      72.899946
lvl2_monitor_concentration         30.417335
lvl2_assessment_concentration      30.417335
lvl3_monitor_concentration         48.903157
lvl3_assessment_concentration      48.903157
lvl4_monitor_concentration         34.724452
lvl4_assessment_concentration      34.751204
lvl5_monit

# Missing Data Handling

In [8]:
def handle_missing_values(airqualitydata):
    """
    Handle missing values in a DataFrame with proper type checking
    """
    airqualitydata_clean = airqualitydata.copy()
    
    # 1. Drop high-missing columns
    cols_to_drop = [
        'auditing_agency_code', 'auditing_agency',
        'tribal_code', 'tribe_name',
        'lvl10_monitor_concentration', 'lvl10_assessment_concentration',
        'lvl9_monitor_concentration', 'lvl9_assessment_concentration'
    ]
    cols_to_drop = [col for col in cols_to_drop if col in airqualitydata_clean.columns]
    airqualitydata_clean = airqualitydata_clean.drop(columns=cols_to_drop)
    
    # 2. Process each column with type-specific handling
    for col in airqualitydata_clean.columns:
        if airqualitydata_clean[col].isna().sum() > 0:  # Only process columns with missing values
            try:
                # For numeric columns
                if pd.api.types.is_numeric_dtype(airqualitydata_clean[col]):
                    # Create missing indicator for concentration columns
                    if 'concentration' in col:
                        airqualitydata_clean[f'{col}_missing'] = airqualitydata_clean[col].isna().astype(int)
                    # Fill with median for numeric
                    median_val = airqualitydata_clean[col].median()
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(median_val)
                
                # For string/object columns
                elif pd.api.types.is_string_dtype(airqualitydata_clean[col]):
                    mode_val = airqualitydata_clean[col].mode()[0] if not airqualitydata_clean[col].mode().empty else 'Unknown'
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(mode_val)
                
                # For datetime columns
                elif pd.api.types.is_datetime64_any_dtype(airqualitydata_clean[col]):
                    airqualitydata_clean[col] = airqualitydata_clean[col].fillna(method='ffill')
                
            except Exception as e:
                print(f"Could not process column {col}: {str(e)}")
                # Fallback to simple fill for problematic columns
                airqualitydata_clean[col] = airqualitydata_clean[col].fillna('Unknown') if pd.api.types.is_string_dtype(airqualitydata_clean[col]) else airqualitydata_clean[col].fillna(0)
    
    return airqualitydata_clean

In [9]:
# Process missing values
airqualitydata_clean = handle_missing_values(airqualitydata)

In [10]:
#percentage of missing value
airqualitydata_clean.isnull().sum()/airqualitydata_clean.shape[0]*100

index                                    0.0
state_code                               0.0
county_code                              0.0
site_number                              0.0
parameter_code                           0.0
                                        ... 
lvl6_assessment_concentration_missing    0.0
lvl7_monitor_concentration_missing       0.0
lvl7_assessment_concentration_missing    0.0
lvl8_monitor_concentration_missing       0.0
lvl8_assessment_concentration_missing    0.0
Length: 67, dtype: float64

# Data Transformation and Feature Engineering

In [12]:
airqualitydata_clean.head()

,index,state_code,county_code,site_number,parameter_code,poc,parameter,si_id,method_code,method,...,lvl4_monitor_concentration_missing,lvl4_assessment_concentration_missing,lvl5_monitor_concentration_missing,lvl5_assessment_concentration_missing,lvl6_monitor_concentration_missing,lvl6_assessment_concentration_missing,lvl7_monitor_concentration_missing,lvl7_assessment_concentration_missing,lvl8_monitor_concentration_missing,lvl8_assessment_concentration_missing
0,0,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
1,1,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
2,2,1,49,9991,44201,1,Ozone,96297,47,INSTRUMENTAL-ULTRA VIOLET,...,0,0,1,1,0,0,1,1,1,1
3,3,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0
4,4,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,0,0,0,0,1,1,1,1,0,0


In [13]:
#Finding unique values in method column
airqualitydata_clean.method.value_counts()

method
INSTRUMENTAL-ULTRA VIOLET ABSORPTION                                                                                   1236
INSTRUMENTAL-ULTRA VIOLET                                                                                               713
INSTRUMENTAL-GAS PHASE CHEMILUMINESCENCE                                                                                265
INSTRUMENTAL-ULTRAVIOLET FLUORESCENCE                                                                                   226
INSTRUMENTAL-PULSED FLUORESCENT                                                                                         166
INSTRUMENTAL-Pulsed Fluorescent 43C-TLE/43i-TLE                                                                         153
Teledyne Model T500U-Cavity Attenuated Phase Shift Spectroscopy                                                         133
INSTRUMENTAL-CHEMILUMINESCENCE                                                                                          125
I

In [14]:
#standardisation of Method column ( unique method descriptions into analyzable categories using regular expression)
airqualitydata_clean['method_clean'] = (
    airqualitydata_clean['method']
    .str.replace(
        r'(?i)(instrumental|analyzer)[\s\-]*(ultra[\s\-]*violet|uv)', 
        'UV', 
        regex=True
    )
    .str.replace(r'(?i)\b(absorption|spectrometry|analysis)\b', '', regex=True)
    .str.strip()
    .replace('', 'UV Method')
)

In [15]:

# Create Composite Site ID
airqualitydata_clean['site_id'] = (
    airqualitydata_clean['state_code'].astype(str).str.zfill(2) + '-' +
    airqualitydata_clean['county_code'].astype(str).str.zfill(3) + '-' +
    airqualitydata_clean['site_number'].astype(str).str.zfill(4)
)

# Extract Measurement Precision from Method
airqualitydata_clean['precision_code'] = (
    airqualitydata_clean['method']
    .str.extract(r'(\bCLASS\s*[IVX]+|\bPRECISION\s*\d+)', flags=re.IGNORECASE)
    .squeeze()
)

In [16]:
#Flag Incomplete Monitoring Data
missing_cols = [col for col in airqualitydata_clean.columns if 'missing' in col]
airqualitydata_clean['data_completeness'] = 1 - (airqualitydata_clean[missing_cols].sum(axis=1) / len(missing_cols))

In [17]:
# Categorize by Data Quality
conditions = [
    airqualitydata_clean['data_completeness'] >= 0.9,
    airqualitydata_clean['data_completeness'] >= 0.5,
]
choices = ['High', 'Medium']
airqualitydata_clean['quality_category'] = np.select(conditions, choices, default='Low')


In [18]:
# Classify Agency Type 
airqualitydata_clean['agency_type'] = (
    airqualitydata_clean['monitoring_agency_code']
    .astype(str)
    .str.extract(r'^([A-Z]+)')[0]
    .replace({'EPA': 'Federal', 'STATE': 'State', None: 'Local'})
)

In [19]:
# Pre-format Display Text 
airqualitydata_clean['display_text'] = (
    "Site " + airqualitydata_clean['site_number'].astype(str) + 
    " (" + airqualitydata_clean['parameter'] + "): " + 
    airqualitydata_clean['lvl1_monitor_concentration'].round(2).astype(str) + " " + 
    airqualitydata_clean['unit'].fillna('ppm')
)


In [20]:
# Target encoding for counties
county_means = airqualitydata_clean.groupby('county_code')['lvl1_monitor_concentration'].mean().to_dict()
airqualitydata_clean['county_encoded'] = airqualitydata_clean['county_code'].map(county_means)

In [21]:
# Frequency encoding for parameters
param_freq = airqualitydata_clean['parameter'].value_counts(normalize=True).to_dict()
airqualitydata_clean['param_freq_encoded'] = airqualitydata_clean['parameter'].map(param_freq)


In [22]:
# Convert to datetime if not already
airqualitydata_clean['date_of_last_change'] = pd.to_datetime(airqualitydata_clean['date_of_last_change'])


In [23]:
# Extract temporal components
airqualitydata_clean['change_year'] = airqualitydata_clean['date_of_last_change'].dt.year
airqualitydata_clean['change_quarter'] = airqualitydata_clean['date_of_last_change'].dt.quarter
airqualitydata_clean['days_since_change'] = (pd.Timestamp.now() - airqualitydata_clean['date_of_last_change']).dt.days

In [24]:
# Cyclical encoding for seasons
airqualitydata_clean['change_month_sin'] = np.sin(2*np.pi*airqualitydata_clean['date_of_last_change'].dt.month/12)
airqualitydata_clean['change_month_cos'] = np.cos(2*np.pi*airqualitydata_clean['date_of_last_change'].dt.month/12)

In [25]:
# Method complexity (word count)
airqualitydata_clean['method_complexity'] = airqualitydata_clean['method'].str.split().str.len()

In [26]:
# Equipment type from method text
airqualitydata_clean['equipment_type'] = np.where(
    airqualitydata_clean['method'].str.contains('INSTRUMENTAL', case=False), 
    'Analytical', 
    'Manual'
)

In [27]:

# Season-concentration interaction
airqualitydata_clean['summer_highs'] = (airqualitydata_clean['date_of_last_change'].dt.month.isin([6,7,8])) & (airqualitydata_clean['lvl1_monitor_concentration'] > 60)


In [28]:
# Site-level percentiles
airqualitydata_clean['site_percentile'] = airqualitydata_clean.groupby('site_number')['lvl1_monitor_concentration'].rank(pct=True)


In [29]:

# County-level volatility
airqualitydata_clean['county_volatility'] = airqualitydata_clean.groupby(['county_code','change_year'])['lvl1_monitor_concentration'].transform('std')


In [30]:
# Final datetime conversion
airqualitydata_clean['date'] = pd.to_datetime(airqualitydata_clean['date_of_last_change'])

In [31]:
# Data preprocessing
numeric_cols = ['lvl1_monitor_concentration', 'county_volatility', 'site_percentile']
airqualitydata_clean[numeric_cols] = airqualitydata_clean[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# Visualisation 

In [33]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import dash_bootstrap_components as dbc

# Initialize Dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
server = app.server

app.layout = dbc.Container([
    # Title
    dbc.Row(dbc.Col(html.H1("Air Quality Dashboard", className="text-center my-4"))),
    
    # Filters
    dbc.Row([
        dbc.Col([
            html.Label("Select States:"),
            dcc.Dropdown(
                id='state-filter',
                options=[{'label': state, 'value': state} 
                        for state in sorted(airqualitydata_clean['state_name'].unique())],
                multi=True,
                placeholder='All States'
            )
        ], md=3),
        
        dbc.Col([
            html.Label("Select Parameter:"),
            dcc.Dropdown(
                id='parameter-filter',
                options=[{'label': p, 'value': p} for p in airqualitydata_clean['parameter'].unique()],
                value='Ozone'
            )
        ], md=3),
        
        dbc.Col([
            html.Label("Data Quality:"),
            dcc.Dropdown(
                id='quality-filter',
                options=[{'label': q, 'value': q} for q in ['High', 'Medium', 'Low']],
                multi=True,
                placeholder='All Quality Levels'
            )
        ], md=3),
        
        dbc.Col([
            html.Label("Date Range:"),
            dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=airqualitydata_clean['date'].min(),
                max_date_allowed=airqualitydata_clean['date'].max(),
                start_date=airqualitydata_clean['date'].min(),
                end_date=airqualitydata_clean['date'].max()
            )
        ], md=3)
    ], className="mb-4"),
    
    # Main visualizations
    dbc.Row([
        dbc.Col(dcc.Graph(id='geo-map'), md=6),
        dbc.Col([
            dcc.Graph(id='state-time-trend'),
            dcc.Graph(id='method-analysis')
        ], md=6)
    ], className="mb-4"),
    
    # Second row of visualizations
    dbc.Row([
        dbc.Col(dcc.Graph(id='heatmap'), md=6),
        dbc.Col(dcc.Graph(id='state-comparison-box'), md=6)
    ], className="mb-4"),
    
    # Third row
    dbc.Row([
        dbc.Col(dcc.Graph(id='equipment-analysis'), md=6),
        dbc.Col(dcc.Graph(id='quality-analysis'), md=6)
    ], className="mb-4")
], fluid=True)

# Callbacks
@app.callback(
    [Output('geo-map', 'figure'),
     Output('state-time-trend', 'figure'),
     Output('state-comparison-box', 'figure'),
     Output('heatmap', 'figure'),
     Output('method-analysis', 'figure'),
     Output('equipment-analysis', 'figure'),
     Output('quality-analysis', 'figure')],
    [Input('state-filter', 'value'),
     Input('parameter-filter', 'value'),
     Input('quality-filter', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date')]
)
def update_dashboard(selected_states, selected_param, selected_quality, start_date, end_date):
    # Filter data
    filtered_df = airqualitydata_clean.copy()
    
    if selected_param:
        filtered_df = filtered_df[filtered_df['parameter'] == selected_param]
    
    if selected_states:
        filtered_df = filtered_df[filtered_df['state_name'].isin(selected_states)]
    
    if selected_quality:
        filtered_df = filtered_df[filtered_df['quality_category'].isin(selected_quality)]
    
    filtered_df = filtered_df[
        (filtered_df['date'] >= start_date) & 
        (filtered_df['date'] <= end_date)
    ]
    
    if filtered_df.empty:
        empty_fig = px.scatter(title="No data available for selected filters")
        return [empty_fig] * 7
    
    # 1. Map with enhanced hover data
    geo_fig = px.scatter_mapbox(
        filtered_df,
        lat='latitude',
        lon='longitude',
        color='state_name',
        size='lvl1_monitor_concentration',
        hover_name='display_text',
        hover_data=['method_clean', 'quality_category', 'agency_type'],
        zoom=4,
        height=500,
        title='Monitoring Sites with Quality Indicators'
    )
    geo_fig.update_layout(mapbox_style="open-street-map")
    
    # 2. State Time Trend with seasonal markers
    if selected_states:
        trend_df = filtered_df.groupby(['date', 'state_name'])['lvl1_monitor_concentration'].mean().reset_index()
        trend_fig = px.line(
            trend_df,
            x='date',
            y='lvl1_monitor_concentration',
            color='state_name',
            title='Concentration Trend by State with Seasonal Patterns',
            markers=True
        )
    else:
        trend_df = filtered_df.groupby('date')['lvl1_monitor_concentration'].mean().reset_index()
        trend_fig = px.line(
            trend_df,
            x='date',
            y='lvl1_monitor_concentration',
            title='Overall Concentration Trend',
            markers=True
        )
    
    # 3. State Comparison Box Plot with quality categories
    box_fig = px.box(
        filtered_df,
        x='state_name',
        y='lvl1_monitor_concentration',
        color='quality_category',
        title='State Comparison with Quality Categories',
        hover_data=['method_clean']
    )
    
    # 4. Heatmap of concentration by method and state
    heatmap_df = filtered_df.groupby(['method_clean', 'state_name'])['lvl1_monitor_concentration'].mean().reset_index()
    heatmap_fig = px.density_heatmap(
        heatmap_df,
        x='method_clean',
        y='state_name',
        z='lvl1_monitor_concentration',
        title='Concentration by Method and State',
        color_continuous_scale='Viridis',
        height=400
    )
    
    # 5. Method analysis
    method_fig = px.bar(
        filtered_df.groupby('method_clean').agg({
            'lvl1_monitor_concentration': 'mean',
            'site_number': 'nunique'
        }).reset_index(),
        x='method_clean',
        y='lvl1_monitor_concentration',
        color='site_number',
        title='Average Concentration by Method (Colored by Site Count)'
    )
    
    # 6. Equipment analysis
    equip_fig = px.violin(
        filtered_df,
        x='equipment_type',
        y='lvl1_monitor_concentration',
        color='quality_category',
        box=True,
        points="all",
        title='Equipment Type vs Concentration'
    )
    
    # 7. Quality analysis
    quality_fig = px.sunburst(
        filtered_df,
        path=['quality_category', 'agency_type', 'state_name'],
        values='site_number',
        title='Data Quality Distribution'
    )
    
    return geo_fig, trend_fig, box_fig, heatmap_fig, method_fig, equip_fig, quality_fig

if __name__ == '__main__':
        app.run(jupyter_mode='external', port=8056)

Dash app running on http://127.0.0.1:8056/
